In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPEN_API_KEY"]=os.getenv("OPEN_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

## StateBackend

In [5]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [6]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model="google_genai:gemini-2.5-flash-lite")

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite",
    backend=StateBackend(),
)

In [7]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
#    (StateBackend keeps that file inside LangGraph state)
# -----------------------------------------------------------------------------
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
I have created the file /notes/todo.txt with the content you provided.


In [8]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='6d0b424b-f007-4d9e-92eb-a1597634f403'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"file_path": "/notes/todo.txt", "content": "1. Record video\\n2. Edit video\\n3. Upload video"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9e4c-12e7-7503-bada-8945c2344293-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/notes/todo.txt', 'content': '1. Record video\n2. Edit video\n3. Upload video'}, 'id': 'c50450f1-1e17-49a4-af46-1ea21efa8cee', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5944, 'output_tokens': 41, 'total_tokens': 5985, 'input_token_details': {'cache_r

In [9]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# -----------------------------------------------------------------------------
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-06-06T18:57:29.851783+00:00', 'modified_at': '2026-06-06T18:57:29.851783+00:00'}


In [10]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
1. Record video
2. Edit video
3. Upload video


## FileSystemBackend(local disk)

In [13]:
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(
                        model="google_genai:gemini-2.5-flash-lite",
                        backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True)
)
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [14]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
I have created the file at /notes/todo.txt with the content you provided.


In [15]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 C:\Users\cheru\OneDrive\Desktop\GitHub\Langchain_Agent_course\05_deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [16]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
The content of /notes/todo.txt is:
1. Record video
2. Edit video
3. Upload video


## Deep Agent — StoreBackend verification
Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [18]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [19]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)



--- Agent reply (thread 1) --------------------------------------
I've created the file /notes/todo.txt with the content you provided.


In [20]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
The content of /notes/todo.txt is:
1. Record video
2. Edit video
3. Upload video


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

### Deep Agent Backends — Where Your Files Actually Live
Every Deep Agent works with a virtual filesystem: its tools read and write files using paths like /notes/todo.txt. But those paths are an abstraction. The backend decides where that data physically lives — and that single choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

1. **StateBackend** (default)

    Files live in the agent's LangGraph state — i.e. in RAM, tied to a single thread. They're available during the run via result["files"], and you must manually carry that state forward to keep using them.

- Where: in-memory dict, inside one thread's state
- Survives: within a single conversation/thread only
- Gone when: the thread ends or the process exits
- Use it for: ephemeral scratch space, temporary working files

2. **FilesystemBackend**

    Files are written to your real disk, under root_dir. With virtual_mode=True, a virtual path like /notes/todo.txt maps to root_dir/notes/todo.txt. These are genuine files you can open in an editor or cat from a terminal.

- Where: your actual filesystem, relative to root_dir
- Survives: across process restarts — it's a real file
- Use it for: when the agent should edit real project files
- ⚠️ Caution: grants the agent real read/write disk access — point it at a scratch directory, not anything important

3. **StoreBackend**

    Files live in a LangGraph store, scoped by a namespace. This is the only backend whose files persist across threads/conversations — write on one thread, read back on another. With InMemoryStore this works within a single process; swap in a persistent store (e.g. Postgres-backed) for true cross-restart durability.

- Where: inside the store object, under a namespace key
- Survives: across threads sharing the same store
- Gone when: the process exits (if using InMemoryStore) — use a persistent store to outlive restarts
- Use it for: long-term memory, per-user file spaces

#### Quick comparison
    Backend	                Lives in	      Cross-thread?	    Survives restart?	    Real file on disk?
    StateBackend            LangGraph state          ❌                  ❌	                 ❌
    FilesystemBackendYour   disk                     ✅                  ✅	                 ✅
    StoreBackend            A LangGraph store	 ✅              only w/ persistent store	 ❌

Mental model: the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.

